# 세 가지 RAG 파이프라인 비교 노트북 (전체 청약 데이터셋)

이 노트북은 `windows_home_project_driver.ipynb`와 같은 프로젝트(`sct-project-graphrag`)를 사용하되,
**mini corpus가 아니라 전체 청약(분양) 규정·가이드 데이터셋**에 대해 세 파이프라인의 답변을 한 번에 비교합니다.

| | 파이프라인 | 실행 스크립트 | 입력 데이터 |
|---|---|---|---|
| 1 | **Chunk BM25** (baseline) | `scripts_home/09_bm25_raw_rag_answer.py` | `data_home/processed/pdf_pages.jsonl` |
| 2 | **Issue-based** (structural) | `scripts_home/10_bm25_issue_rag_answer.py --raw-context same-case` | `data_home/processed/case_issues.jsonl` (+ pages) |
| 3 | **Graph RAG** (relational) | `scripts_home/14_graph_rag_answer.py` | `data_home/indexes/issue_graph_with_similarity.json` |

세 파이프라인 모두 전처리 산출물이 이미 repo에 들어 있으므로, 아래 **3~5절을 순서대로 한 번만** 실행한 뒤,
마지막 6절에서 `compare("질문")` 셀을 복사해 **질문 문자열만 바꿔가며** 여러 개를 테스트하면 됩니다.

## 1. 프로젝트 폴더 경로 설정

`PROJECT_DIR`만 본인 컴퓨터의 프로젝트 폴더 경로로 고치세요. Windows 경로 앞에는 `r`을 붙입니다.

In [ ]:
from pathlib import Path
import os
import sys

# 예: r"C:\\sct-project-graphrag"  /  r"D:\\snu_python\\graphrag"
PROJECT_DIR = Path(r"D:\snu_python\graphrag")

# 전체 파이프라인이므로 scripts_home/ 와 data_home/ 가 있어야 합니다.
REQUIRED_MARKERS = ["pyproject.toml", "scripts_home", "data_home"]

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"프로젝트 폴더를 찾지 못했습니다: {PROJECT_DIR}")

missing = [m for m in REQUIRED_MARKERS if not (PROJECT_DIR / m).exists()]
if missing:
    raise RuntimeError(
        f"프로젝트 폴더로 보이지 않습니다: {PROJECT_DIR}\n"
        f"없는 항목: {missing}\n"
        "zip이 한 겹 더 들어가서 풀렸는지 확인하고 PROJECT_DIR을 안쪽 폴더로 고치세요."
    )

os.chdir(PROJECT_DIR)
print("프로젝트 폴더:", PROJECT_DIR)
print("현재 작업 폴더:", Path.cwd())
print("노트북 Python:", sys.executable)

## 2. 패키지 설치 (이미 했다면 건너뛰어도 됩니다)

`windows_home_project_driver.ipynb`에서 한 번 설치했다면 다시 실행하지 않아도 됩니다.

In [ ]:
# %pip install --upgrade pip
# %pip install -e ".[graph]"

## 3. API 환경변수 설정

학내 Wi-Fi에 연결된 상태에서 실행하세요. `.env` 파일은 `PROJECT_DIR/.env`에 있어야 하며,
세 스크립트 모두 이 `.env`를 읽습니다.

In [ ]:
def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        raise FileNotFoundError(f".env 파일을 찾지 못했습니다: {env_path}")
    with env_path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")

ENV_FILE = PROJECT_DIR / ".env"
load_env_file(ENV_FILE)

required = ["OPENAI_API_KEY", "BASE_URL", "OPENAI_MODEL", "OPENAI_REASONING_EFFORT", "OPENAI_EMBEDDING_MODEL"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise RuntimeError(f".env에 없는 값: {missing}. {ENV_FILE} 파일을 확인하세요.")

print(f"{ENV_FILE}에서 API 환경변수를 불러왔습니다.")
print("BASE_URL     :", os.environ["BASE_URL"])
print("OPENAI_MODEL :", os.environ["OPENAI_MODEL"])

## 4. 전체 청약 데이터셋 산출물 확인

세 파이프라인이 사용할 전처리 결과가 `data_home/` 아래에 있는지 점검합니다.
repo에는 전체 처리 결과가 이미 포함되어 있으므로, 보통은 아래 셀이 모두 `OK`로 나옵니다.

In [ ]:
DATA = PROJECT_DIR / "data_home"
SCRIPTS_DIR = PROJECT_DIR / "scripts_home"

PDF_PAGES   = DATA / "processed" / "pdf_pages.jsonl"                 # 파이프라인 1, 2
CASE_ISSUES = DATA / "processed" / "case_issues.jsonl"               # 파이프라인 2
SIM_GRAPH   = DATA / "indexes"   / "issue_graph_with_similarity.json"  # 파이프라인 3

def _count_lines(p):
    try:
        with p.open(encoding="utf-8") as f:
            return sum(1 for _ in f)
    except Exception:
        return "?"

required_files = {
    "pdf_pages.jsonl   (P1, P2)": PDF_PAGES,
    "case_issues.jsonl (P2)":     CASE_ISSUES,
    "issue_graph_with_similarity.json (P3)": SIM_GRAPH,
}

all_ok = True
for label, path in required_files.items():
    if path.exists():
        extra = f" lines={_count_lines(path)}" if path.suffix == ".jsonl" else ""
        print(f"OK    {label}{extra}")
    else:
        all_ok = False
        print(f"MISS  {label}  ->  {path}")

if all_ok:
    print("\n전처리 산출물이 모두 준비되었습니다. 5절로 넘어가세요.")
else:
    print("\n누락된 파일이 있습니다. 4-b절(선택)에서 해당 단계를 다시 생성하세요.")

In [ ]:
from sct_graphrag.io import load_jsonl, group_pages_by_document, iter_issue_records
import json
import subprocess

MANIFEST = DATA / "processed" / "sample_cases.jsonl"
PDF_DIR  = DATA / "raw" / "pdfs"

# 1) 매니페스트 / 내려받은 PDF
manifest_rows = load_jsonl(MANIFEST) if MANIFEST.exists() else []
pdf_files = sorted(PDF_DIR.glob("*.pdf")) if PDF_DIR.exists() else []

# 2) 페이지 텍스트 → 문서 수
pages = load_jsonl(PDF_PAGES)
page_docs = group_pages_by_document(pages)

# 3) 쟁점 레코드 → 문서 수
issues = list(iter_issue_records(load_jsonl(CASE_ISSUES)))
issue_docs = {r["document_id"] for r in issues}

# 4) 그래프 → Issue 노드와 연결된 문서 수
raw = json.loads(SIM_GRAPH.read_text(encoding="utf-8"))
nodes = raw.get("nodes", raw) if isinstance(raw, dict) else raw
nodes = list(nodes.values()) if isinstance(nodes, dict) else nodes
from collections import Counter
type_counts = Counter(n.get("type") for n in nodes)
issue_nodes = [n for n in nodes if n.get("type") == "Issue"]
graph_docs = {n.get("document_id") for n in issue_nodes if n.get("document_id")}

print(f"① manifest 문서 수        : {len(manifest_rows)}")
print(f"② 내려받은 PDF 파일 수    : {len(pdf_files)}")
print(f"③ pdf_pages 문서 수       : {len(page_docs)}   (총 페이지 {len(pages)})")
print(f"④ case_issues 문서 수     : {len(issue_docs)}   (총 쟁점 {len(issues)})")
print(f"⑤ graph Issue-연결 문서 수: {len(graph_docs)}   (Issue 노드 {len(issue_nodes)})")
print("  graph 노드 타입 분포     :", dict(type_counts))

# 일치 여부 판정
counts = [len(page_docs), len(issue_docs), len(graph_docs)]
print("\n→ 문서 수 일치 여부:", "OK (정합)" if len(set(counts)) == 1 else f"불일치 {counts} — 가장 작은 단계가 실패한 지점")

# 내용이 깨지지 않았는지 한 건 눈으로 확인
print("\n[샘플 쟁점 레코드]")
s = issues[0]
print("  file     :", s.get("document_id"))
print("  쟁점     :", (s.get("issue") or "")[:80])
print("  결론     :", (s.get("conclusion") or "")[:80])

## 4-b. (선택) 전체 파이프라인 재생성

위 4절에서 누락된 파일이 있거나, 전처리를 처음부터 직접 만들고 싶을 때만 사용합니다.

> ⚠️ `04`(issue 추출)와 `11`(graph feature 추출)은 전체 문서에 대해 LLM을 호출하므로 시간과 API 사용량이 큽니다.
> 수업 중에는 repo에 포함된 처리 결과를 그대로 쓰는 것을 권장합니다.

아래 셀은 기본적으로 **실행되지 않도록** `RUN_FULL_BUILD = False`로 막아두었습니다.
필요한 단계만 선택적으로 켜서 돌리세요.

In [ ]:
# import subprocess

# def run_full(script_name, *args):
#     """scripts_home/ 아래 전체 스크립트를 실행하고 출력을 실시간으로 보여줍니다."""
#     cmd = [sys.executable, str(SCRIPTS_DIR / script_name), *map(str, args)]
#     print("$ " + " ".join(cmd))
#     proc = subprocess.Popen(cmd, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
#                             stderr=subprocess.STDOUT, text=True, bufsize=1)
#     for line in proc.stdout:
#         print(line, end="")
#     proc.wait()
#     if proc.returncode != 0:
#         raise subprocess.CalledProcessError(proc.returncode, cmd)

# RUN_FULL_BUILD = False  # 전체 재생성이 필요할 때만 True 로 바꾸세요.

# if RUN_FULL_BUILD:
#     # 01~03: 메타데이터 -> PDF -> 페이지 텍스트
#     run_full("01_download_case_metadata.py", "--overwrite")
#     run_full("02_download_sample_pdfs.py")
#     run_full("03_extract_pdf_text.py", "--overwrite")
#     # 04~05: issue 추출(LLM) -> issue embedding index  (API 사용, 비용 큼)
#     run_full("04_extract_case_issues.py", "--overwrite", "--concurrency", "3")
#     run_full("05_build_issue_embedding_index.py")
#     # 11~12: graph feature 추출(LLM) -> typed graph + similarity graph  (API 사용, 비용 큼)
#     run_full("11_extract_graph_features.py", "--overwrite", "--concurrency", "3")
#     run_full("12_build_issue_graph.py")
#     run_full("12_build_issue_graph.py", "--add-similarity",
#              "--output", str(SIM_GRAPH))
#     print("\n전체 파이프라인 재생성 완료")
# else:
#     print("RUN_FULL_BUILD=False 이므로 재생성을 건너뜁니다. (repo의 전체 처리 결과 사용)")

## 5. 세 파이프라인 비교 harness

`compare(query)`는 같은 질문을 세 파이프라인에 각각 넣고, 각 스크립트의 `## Answer` 부분을 뽑아 나란히 보여줍니다.
내부적으로는 전체 데이터 경로를 명시적으로 넘겨 실행합니다.

옵션:
- `top_k_raw` : 파이프라인 1에서 가져올 raw chunk 수 (기본 10)
- `top_k_issue` : 파이프라인 2에서 가져올 issue 수 (기본 5)
- `graph_mode` : 파이프라인 3 분석 모드 — `"auto"`(질문에 따라 자동) / `"outcome-comparison"` / `"overview"`
- `show_logs` : `True`면 각 파이프라인이 무엇을 검색했는지(retrieval log)도 함께 표시
- `which` : 일부 파이프라인만 돌리고 싶을 때 예) `which=(1, 3)`

In [ ]:
import subprocess

In [ ]:
import time

PIPELINE_TITLES = {
    1: "파이프라인 1 · Chunk BM25 (baseline)",
    2: "파이프라인 2 · Issue-based (structural)",
    3: "파이프라인 3 · Graph RAG (relational)",
}

def _run(script_name, *args, timeout=900):
    cmd = [sys.executable, str(SCRIPTS_DIR / script_name), *map(str, args)]
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=PROJECT_DIR, capture_output=True, text=True, timeout=timeout)
    elapsed = time.time() - t0
    out = proc.stdout or ""
    if proc.returncode != 0:
        out += "\n\n[stderr]\n" + (proc.stderr or "")
    return proc.returncode, out, elapsed

def _split_answer(stdout):
    """(retrieval_log, answer_text) 로 분리.
    '## Answer' 가 단독 헤더 줄로 처음 나오는 지점에서 자른다.
    답변 본문에 들어 있는 ###/## 헤더에 오작동하지 않도록 줄 단위로 매칭."""
    lines = stdout.splitlines()
    for i, line in enumerate(lines):
        if line.strip() == "## Answer":
            log = "\n".join(lines[:i]).strip()
            answer = "\n".join(lines[i + 1:]).strip()
            return log, answer
    return stdout.strip(), ""

def _run_pipeline(n, query, top_k_raw, top_k_issue, graph_mode):
    if n == 1:
        return _run("09_bm25_raw_rag_answer.py", query,
                    "--input", PDF_PAGES, "--top-k", str(top_k_raw),
                    "--env-file", ENV_FILE)
    if n == 2:
        return _run("10_bm25_issue_rag_answer.py", query,
                    "--issues", CASE_ISSUES, "--pages", PDF_PAGES,
                    "--top-k", str(top_k_issue), "--raw-context", "same-case",
                    "--env-file", ENV_FILE)
    if n == 3:
        return _run("14_graph_rag_answer.py", query,
                    "--graph", SIM_GRAPH, "--analysis-mode", graph_mode,
                    "--env-file", ENV_FILE)
    raise ValueError(n)

def _render(query, results, show_logs):
    try:
        from IPython.display import Markdown, display
        use_md = True
    except Exception:
        use_md = False

    if use_md:
        md = [f"### 질문\n\n> {query}\n"]
        for n, log, answer, elapsed, code in results:
            status = "✅" if code == 0 else "⚠️"
            md.append(f"### {status} {PIPELINE_TITLES[n]}  · _{elapsed:.1f}s_\n\n{answer or '(답변 없음)'}\n")
            if show_logs and log:
                snippet = log[:6000]
                md.append("<details><summary>retrieval log 보기</summary>\n\n```\n"
                          + snippet + "\n```\n</details>\n")
            md.append("\n---\n")
        display(Markdown("\n".join(md)))
    else:
        print("=" * 90)
        print("질문:", query)
        for n, log, answer, elapsed, code in results:
            print("\n" + "=" * 90)
            print(f"{PIPELINE_TITLES[n]}  ({elapsed:.1f}s, exit={code})")
            print("-" * 90)
            if show_logs and log:
                print(log[:4000]); print("-" * 90)
            print(answer or "(답변 없음)")

def compare(query, *, top_k_raw=10, top_k_issue=5, graph_mode="auto",
            show_logs=False, which=(1, 2, 3)):
    results = []
    for n in which:
        print(f"\u25b6 {PIPELINE_TITLES[n]} 실행 중...", flush=True)
        try:
            code, out, elapsed = _run_pipeline(n, query, top_k_raw, top_k_issue, graph_mode)
            log, answer = _split_answer(out)
            results.append((n, log, answer, elapsed, code))
        except Exception as e:
            results.append((n, "", f"[실행 실패] {e!r}", 0.0, -1))
    _render(query, results, show_logs)
    return results

print("compare() 준비 완료")

## 5-b. Retrieval 성능 지표 (label-free)

`compare(...)`가 돌려준 결과만 가지고, 정답 라벨 없이 바로 계산할 수 있는 지표 3가지를 봅니다.

- **score 분포 / 확신도** : top-1 score가 top-k 평균보다 얼마나 높은지 (margin이 클수록 검색이 한 사건에 "확신"을 가졌다는 뜻)
- **사건 겹침도 (Jaccard)** : 세 파이프라인이 찾아온 사건(파일) 집합이 서로 얼마나 겹치는지. 겹침이 낮으면 같은 질문에도 완전히 다른 사건을 근거로 답했다는 뜻
- **Latency** : 파이프라인별 실행 시간 (`compare()`가 이미 측정한 `elapsed`를 표로 정리)

`retrieval log`(각 스크립트의 `## Retrieved ...` / `## Representative Cases` 출력)를 정규식으로 파싱해서 `(score, file_id)` 쌍을 뽑습니다.
별도 정답 데이터가 필요 없어서 모든 쿼리에 바로 적용할 수 있습니다.

In [ ]:
import re
from statistics import mean

# 파이프라인별 retrieval log 라인 포맷
#   P1 (09_bm25_raw_rag_answer.py) : "1. score=0.123 file=foo.pdf chunk=2"
#   P2 (10_bm25_issue_rag_answer.py): "1. score=0.123 file=foo.pdf issue=0"
#   P3 (14_graph_rag_answer.py)     : "  [A1] file=foo.pdf issue=0 outcome=해당 score=12.34"
_P1_P2_LINE = re.compile(r"score=([\d.]+)\s+file=(\S+?)(?:\.pdf)?\s+(?:chunk|issue)=")
_P3_LINE = re.compile(r"\[[A-Z]\d+\]\s+file=(\S+?)(?:\.pdf)?\s+issue=\S+\s+outcome=\S+\s+score=([\d.]+)")


def _parse_retrieval_log(n, log):
    # log 텍스트에서 (score, case_id) 쌍 리스트를 뽑는다. 못 찾으면 빈 리스트.
    pairs = []
    if not log:
        return pairs
    if n in (1, 2):
        for m in _P1_P2_LINE.finditer(log):
            score, case_id = float(m.group(1)), m.group(2)
            pairs.append((score, case_id))
    elif n == 3:
        for m in _P3_LINE.finditer(log):
            case_id, score = m.group(1), float(m.group(2))
            pairs.append((score, case_id))
    return pairs


def _score_stats(pairs, top_k=5):
    # top-1 score, top-k 평균, margin(top1 - top-k 평균)을 계산
    if not pairs:
        return {"top1": None, "topk_mean": None, "margin": None, "n": 0}
    scores = [s for s, _ in pairs[:top_k]]
    top1 = scores[0]
    topk_mean = mean(scores)
    margin = top1 - topk_mean
    return {"top1": round(top1, 3), "topk_mean": round(topk_mean, 3), "margin": round(margin, 3), "n": len(pairs)}


def _jaccard(set_a, set_b):
    if not set_a and not set_b:
        return None
    union = set_a | set_b
    if not union:
        return None
    return len(set_a & set_b) / len(union)


def compute_metrics(results, top_k=5, show_log_excerpt=False):
    # results: compare()가 반환한 [(n, log, answer, elapsed, code), ...]
    parsed = {}
    for n, log, answer, elapsed, code in results:
        pairs = _parse_retrieval_log(n, log)
        case_ids = {case_id for _, case_id in pairs}
        parsed[n] = {
            "pairs": pairs,
            "case_ids": case_ids,
            "elapsed": elapsed,
            "score_stats": _score_stats(pairs, top_k=top_k),
        }
        if show_log_excerpt:
            print(f"-- P{n} retrieval log 일부 (파싱된 {len(pairs)}건) --")
            print((log or "")[:400])
            print()

    try:
        from IPython.display import Markdown, display
        use_md = True
    except Exception:
        use_md = False

    lines = []
    lines.append("| 파이프라인 | top-1 score | top-{0} 평균 | margin | 검색 건수 | latency(s) |".format(top_k))
    lines.append("|---|---|---|---|---|---|")
    for n in sorted(parsed):
        s = parsed[n]["score_stats"]
        lines.append(
            f"| P{n} | {s['top1']} | {s['topk_mean']} | {s['margin']} | {s['n']} | {parsed[n]['elapsed']:.1f} |"
        )
    score_table = "\n".join(lines)

    overlap_lines = ["| 비교 | 겹친 사건 수 | Jaccard |", "|---|---|---|"]
    ns = sorted(parsed)
    for i in range(len(ns)):
        for j in range(i + 1, len(ns)):
            a, b = ns[i], ns[j]
            set_a, set_b = parsed[a]["case_ids"], parsed[b]["case_ids"]
            jac = _jaccard(set_a, set_b)
            inter = len(set_a & set_b)
            jac_str = f"{jac:.2f}" if jac is not None else "n/a"
            overlap_lines.append(f"| P{a} ∩ P{b} | {inter} | {jac_str} |")
    overlap_table = "\n".join(overlap_lines)

    if use_md:
        display(Markdown("**Score 분포 / 확신도**\n\n" + score_table))
        display(Markdown("**사건 겹침도 (Jaccard)**\n\n" + overlap_table))
    else:
        print("Score 분포 / 확신도")
        print(score_table)
        print()
        print("사건 겹침도 (Jaccard)")
        print(overlap_table)

    return parsed


print("compute_metrics() 준비 완료")

## 6. 쿼리 테스트

아래 셀들은 세 가지 질문 유형(파이프라인 1 지향 / 파이프라인 2 지향 / 파이프라인 3 지향)을 예시로 넣어 둔 것입니다.

**셀을 복사한 뒤 `compare("...")` 안의 질문 문자열만 바꿔서** 원하는 만큼 테스트하세요.
각 파이프라인이 어떤 질문에서 강점을 보이는지 비교하면 됩니다.

세 질문 모두 `show_logs=True`로 실행해 retrieval log까지 함께 확인하고, `results1/2/3`에 저장합니다.
이 결과는 7절에서 비교 리포트(.md)로 묶어서 내보냅니다.

### 6-1. 파이프라인 1 지향 질문 (Chunk BM25가 비교적 잘 잡는 유형)

질문에 들어간 수치·조건(전용면적, 지역, 순위)이 규정 원문에 그대로 등장하는 키워드형 질문입니다.

In [ ]:
results1 = compare("서울에 거주하는 사람이 전용면적 102제곱미터 이하 민영주택에 1순위로 청약하려면 청약통장 예치금이 정확히 얼마 있어야 해?", show_logs=True)

### 6-2. 파이프라인 2 지향 질문 (issue 구조화가 유리한 유형)

법률 용어 대신 본인의 상황(세대 구성, 부양가족 자격)을 풀어서 묻기 때문에,
개별 조항을 그대로 찾기보다 쟁점·요건 단위로 구조화된 검색이 유리합니다.

In [ ]:
results2 = compare("본인이 세대주이고 만 65세인 아버지와 62세인 어머니를 4년째 같은 주민등록표에 모시고 있습니다. 그런데 아버지가 본인 명의의 아파트를 한 채 소유하고 계십니다. 가점제 청약 시 부모님 두 분 다 제 부양가족으로 점수를 받을 수 있나요?", show_logs=True)

### 6-3. 파이프라인 3 지향 질문 (Graph RAG가 필요한 유형)

개별 조항 하나가 아니라, 지역·주택유형·규제수준에 따라 결과가 달라지는 패턴을 비교하는 질문입니다.
이런 질문은 `graph_mode="auto"`가 자동으로 패턴 비교 모드를 고릅니다.

In [ ]:
results3 = compare("현재 2주택을 소유한 다주택자입니다. 다주택자라는 이유로 1순위 청약이 원천적으로 차단되는 지역이나 주택 유형은 무엇이며, 반대로 다주택자도 1순위 당첨을 노려볼 수 있는 규제 패턴(지역 및 추첨제 비율)은 어떻게 연결되어 있나요?", show_logs=True)

### 6-4. 성능지표 비교

`results1/2/3`(위 6-1~6-3에서 이미 `show_logs=True`로 받아둔 결과)에 대해
`compute_metrics(...)`로 score 분포·확신도와 사건 겹침도(Jaccard)를 계산합니다.

In [ ]:
metrics1 = compute_metrics(results1, show_log_excerpt=True)

In [ ]:
metrics2 = compute_metrics(results2, show_log_excerpt=True)

In [ ]:
metrics3 = compute_metrics(results3, show_log_excerpt=True)

### 6-5. 빈 템플릿 — 여기에 질문만 바꿔서 계속 테스트 가능

In [ ]:
# 특정 파이프라인만 비교하고 싶을 때 (예: 2번과 3번만)
# results = compare("여기에 질문을 넣으세요", which=(2, 3))
# compute_metrics(results)

In [ ]:
# top-k 와 graph 모드를 바꿔서 실험
# results = compare("여기에 질문을 넣으세요", top_k_raw=15, top_k_issue=8, graph_mode="overview")
# compute_metrics(results)

## 7. 비교 리포트 내보내기 (.md)

6절에서 만든 `results1/2/3`(파이프라인별 답변 + retrieval log)과
`metrics1/2/3`(score 분포, Jaccard 겹침도, latency)을 질문 하나당 한 섹션씩 묶어
**하나의 markdown 파일**로 저장합니다.

- 질문 1 (P1 지향) → `results1`, `metrics1`
- 질문 2 (P2 지향) → `results2`, `metrics2`
- 질문 3 (P3 지향) → `results3`, `metrics3`

먼저 6절의 6-1~6-4 셀을 한 번씩 실행해서 세 변수가 모두 채워져 있어야 합니다.

In [ ]:
from datetime import datetime

OUTPUT_DIR = PROJECT_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = OUTPUT_DIR / "three_pipeline_comparison_report.md"

QUERY_LABELS = {
    1: "질문 1 · 파이프라인 1(Chunk BM25) 지향 — 수치 조건 확인형",
    2: "질문 2 · 파이프라인 2(Issue-based) 지향 — 개인 상황 상담형",
    3: "질문 3 · 파이프라인 3(Graph RAG) 지향 — 규제 패턴 비교형",
}


def _metrics_to_markdown(parsed, top_k=5):
    """compute_metrics()가 반환한 dict(parsed)를 score 표 + Jaccard 표 markdown으로 변환."""
    lines = []
    lines.append(f"| 파이프라인 | top-1 score | top-{top_k} 평균 | margin | 검색 건수 | latency(s) |")
    lines.append("|---|---|---|---|---|---|")
    for n in sorted(parsed):
        s = parsed[n]["score_stats"]
        lines.append(
            f"| P{n} | {s['top1']} | {s['topk_mean']} | {s['margin']} | {s['n']} | {parsed[n]['elapsed']:.1f} |"
        )
    score_table = "\n".join(lines)

    overlap_lines = ["| 비교 | 겹친 사건 수 | Jaccard |", "|---|---|---|"]
    ns = sorted(parsed)
    for i in range(len(ns)):
        for j in range(i + 1, len(ns)):
            a, b = ns[i], ns[j]
            set_a, set_b = parsed[a]["case_ids"], parsed[b]["case_ids"]
            jac = _jaccard(set_a, set_b)
            inter = len(set_a & set_b)
            jac_str = f"{jac:.2f}" if jac is not None else "n/a"
            overlap_lines.append(f"| P{a} ∩ P{b} | {inter} | {jac_str} |")
    overlap_table = "\n".join(overlap_lines)

    return score_table, overlap_table


def _results_to_markdown(query_no, query, results, metrics):
    """질문 하나에 대한 results(답변+log) + metrics(성능지표)를 하나의 markdown 섹션으로 변환."""
    md = []
    md.append(f"## {QUERY_LABELS.get(query_no, f'질문 {query_no}')}")
    md.append("")
    md.append(f"**질문:** {query}")
    md.append("")

    for n, log, answer, elapsed, code in results:
        status = "성공" if code == 0 else "실패"
        md.append(f"### {PIPELINE_TITLES[n]} ({status}, {elapsed:.1f}s)")
        md.append("")
        md.append(answer or "(답변 없음)")
        md.append("")
        if log:
            md.append("<details><summary>retrieval log</summary>")
            md.append("")
            md.append("```")
            md.append(log[:6000])
            md.append("```")
            md.append("")
            md.append("</details>")
            md.append("")

    score_table, overlap_table = _metrics_to_markdown(metrics)
    md.append("### 성능 지표")
    md.append("")
    md.append("**Score 분포 / 확신도**")
    md.append("")
    md.append(score_table)
    md.append("")
    md.append("**사건 겹침도 (Jaccard)**")
    md.append("")
    md.append(overlap_table)
    md.append("")
    md.append("---")
    md.append("")
    return "\n".join(md)


def build_comparison_report(queries, results_list, metrics_list, output_path=REPORT_PATH):
    """질문/결과/성능지표를 받아 하나의 .md 리포트로 저장.

    queries      : [query1, query2, query3]
    results_list : [results1, results2, results3]  (compare()의 반환값)
    metrics_list : [metrics1, metrics2, metrics3]   (compute_metrics()의 반환값)
    """
    sections = []
    sections.append("# 세 가지 RAG 파이프라인 비교 리포트 (청약 시장 Q&A)")
    sections.append("")
    sections.append(f"생성 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    sections.append("")
    sections.append(
        "| | 파이프라인 | 방식 |\n|---|---|---|\n"
        "| P1 | Chunk BM25 (baseline) | 원문 chunk 검색 |\n"
        "| P2 | Issue-based (structural) | 쟁점/요건 구조화 검색 |\n"
        "| P3 | Graph RAG (relational) | 이슈 그래프 기반 패턴 검색 |"
    )
    sections.append("")
    sections.append("---")
    sections.append("")

    for i, (query, results, metrics) in enumerate(zip(queries, results_list, metrics_list), start=1):
        sections.append(_results_to_markdown(i, query, results, metrics))

    report_text = "\n".join(sections)
    output_path.write_text(report_text, encoding="utf-8")
    print(f"리포트 저장 완료: {output_path}  ({len(report_text)} chars)")
    return output_path


report_path = build_comparison_report(
    queries=[
        "서울에 거주하는 사람이 전용면적 102제곱미터 이하 민영주택에 1순위로 청약하려면 청약통장 예치금이 정확히 얼마 있어야 해?",
        "본인이 세대주이고 만 65세인 아버지와 62세인 어머니를 4년째 같은 주민등록표에 모시고 있습니다. 그런데 아버지가 본인 명의의 아파트를 한 채 소유하고 계십니다. 가점제 청약 시 부모님 두 분 다 제 부양가족으로 점수를 받을 수 있나요?",
        "현재 2주택을 소유한 다주택자입니다. 다주택자라는 이유로 1순위 청약이 원천적으로 차단되는 지역이나 주택 유형은 무엇이며, 반대로 다주택자도 1순위 당첨을 노려볼 수 있는 규제 패턴(지역 및 추첨제 비율)은 어떻게 연결되어 있나요?",
    ],
    results_list=[results1, results2, results3],
    metrics_list=[metrics1, metrics2, metrics3],
)